Copyright 2021 The TensorFlow Authors.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required bsteps_per_epoch y applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.steps_per_epoch 
# limitations under the License.

#Assess privacy risks with TensorFlow Privacy Secret Sharer Attack

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/tensorflow/privacy/blob/master/tensorflow_privacy/privacy/privacy_tests/secret_sharer/secret_sharer_example.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/tensorflow/privacy/blob/master/tensorflow_privacy/privacy/privacy_tests/secret_sharer/secret_sharer_example.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>

In this colab, we demonstrate how [secret sharer](https://arxiv.org/abs/1802.08232) works on a language model. We will train a model with "secrets", i.e. random sequences, inserted in the training data, and then evaluate if the model has "memorized" those secrets.

(The code is largely based on a Tensorflow tutorial for training a character-level LSTM on Shakespeare dataset.)

#Setup

You may set the runtime to use a GPU by Runtime > Change runtime type > Hardware accelerator.

In [ ]:

%pip install tf-models-official
#!pip install git+https://github.com/tensorflow/privacy.git
#clear_output()

In [78]:
import functools
import os
import numpy as np
import tensorflow as tf


from tensorflow_privacy.privacy.privacy_tests.secret_sharer.generate_secrets import SecretConfig, TextSecretProperties, generate_text_secrets_and_references, construct_secret
from tensorflow_privacy.privacy.privacy_tests.secret_sharer.exposures import compute_exposure_interpolation, compute_exposure_extrapolation

# Define functions

In [79]:


SEQ_LENGTH = 1024

BATCHES_PER_EPOCH = 200
TRAIN_EPOCHS = 2
BATCH_SIZE = 16
DATASET_SIZE = 20000




In [80]:
pip install pandas

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [81]:
import pandas as pd
import csv
def load_text_from_csv(path_to_file):
    """Carga el texto de la columna 'TEXT' de un archivo CSV."""
    df = pd.read_csv(path_to_file, usecols=["TEXT"], dtype=str, encoding="ISO-8859-1", quoting=csv.QUOTE_NONE) 
    text = " ".join(df["TEXT"].dropna().iloc[:DATASET_SIZE*2].tolist()) 
    return text

In [88]:
#@title Function for getting dataset combined with secrets.
def get_dataset(secrets=None,
                batch_size=None, seq_length=SEQ_LENGTH):
  """Creates a dataset from a given text file.

  Args:
    secrets: Secrets to be inserted.
    batch_size: Batch size to use.
    seq_length: The length of the LSTM sequence.

  Returns:
    A tuple, consisting of the Dataset and the class to character mapping
    and character to class mapping.
  """
  tokenizer = AutoTokenizer.from_pretrained("gpt2")
  tokenizer.pad_token = tokenizer.eos_token
  text = load_text_from_csv("MIMIC_NoteEvents_deanonymized.csv")

  # Create vocab
  vocab = sorted(set(text))
  char2idx = {u: i for i, u in enumerate(vocab)}
  idx2char = np.array(vocab)

  # Gather all secrets
  if not secrets:
    pass
  else:
    all_secrets = []
    for secret in secrets:
      secret.datasets = {}
      for r, seqs in secret.secrets.items():
        all_secrets += seqs * r
        indexed_seqs = np.array([tokenizer.encode(s, padding='max_length', max_length=seq_length) for s in seqs])
        secret.datasets[r] = {
            'data': tf.data.Dataset.from_tensor_slices(indexed_seqs).map(lambda chunk: chunk[:-1]).batch(1),
            #'label': list(map(lambda chunk: chunk[1:], indexed_seqs))
            'label': list(map(lambda chunk: np.roll(chunk, -1), indexed_seqs))
        }
      indexed_seqs = np.array([tokenizer.encode(s,padding='max_length', max_length=seq_length) for s in secret.references])
      secret.datasets[0] = {
          'data': tf.data.Dataset.from_tensor_slices(indexed_seqs).map(lambda chunk: chunk[:-1]).batch(1),
          #'label': list(map(lambda chunk: chunk[1:], indexed_seqs))
          'label': list(map(lambda chunk: np.roll(chunk, -1), indexed_seqs))
      }
    # Insert the secrets into the training data
    idx = np.random.choice(len(text) - 1, len(all_secrets) - 1, replace=True) + 1
    # split text into len(all_secrets) parts
    text = np.split(list(text), np.sort(idx))
    text = [''.join(t) for t in text]
    text = ''.join(np.vstack([text, all_secrets]).transpose().reshape(-1))

  # Split text into sequence length + 1 chucks to create examples
  def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text


  tokenizer = AutoTokenizer.from_pretrained("gpt2")
  tokenizer.pad_token = tokenizer.eos_token
  input_ids = tokenizer.encode(text, return_tensors="tf")[0]
  seq_length = 1024  # GPT-2 usa una secuencia máxima de 1024 tokens
  dataset = tf.data.Dataset.from_tensor_slices(input_ids)
  dataset = dataset.batch(seq_length + 1, drop_remainder=True)
  def split_input_target(chunk):
    return chunk[:-1], chunk[1:]

  dataset = dataset.map(split_input_target)
  dataset = dataset.shuffle(10000)
    

  train_dataset = dataset.take(2000).batch(8, drop_remainder=True)
  val_dataset = dataset.skip(2000).batch(8, drop_remainder=True)
  return train_dataset, val_dataset

In [ ]:
from tensorflow_privacy.privacy.analysis import compute_noise_from_budget

epsilon = 10
delta = 1 / DATASET_SIZE
l2_norm_clip = 1.0 
# Calculamos el noise_multiplier para garantizar (ε, δ)-DP
noise_multiplier = compute_noise_from_budget(
    n=DATASET_SIZE,  # Tamaño del dataset
    batch_size=BATCH_SIZE,
    epsilon=epsilon,
    epochs=train_epochs,
    delta=delta,
    min_noise=1e-6
)

In [83]:
import os
import tensorflow as tf
from transformers import TFAutoModelForCausalLM, create_optimizer, AutoTokenizer, AdamWeightDecay
from tensorflow.keras.optimizers import SGD
from tensorflow_privacy.privacy.optimizers.dp_optimizer_keras import DPKerasSGDOptimizer
from tensorflow.keras.callbacks import History

def train_model(dataset, validation_dataset, batch_size, vocab_size, train_epochs, checkpoint_dir, dp):
    """Entrena un modelo Transformer con checkpoints."""

    train_steps = BATCHES_PER_EPOCH // batch_size 

    model = TFAutoModelForCausalLM.from_pretrained("gpt2")
    noise_multiplier = 0.4272711315437153
    l2_norm_clip = 1e-6
    num_microbatches = BATCH_SIZE
    if dp:
        optimizer = DPKerasSGDOptimizer(
            learning_rate=0.03,
            noise_multiplier=noise_multiplier,
            l2_norm_clip=l2_norm_clip,
            num_microbatches=1
        )
        print("🔒 Usando DP-SGD con noise_multiplier =", noise_multiplier)
    else:
        optimizer = SGD(learning_rate=0.03)

    model.compile(optimizer=optimizer, loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), jit_compile=True)


    callbacks = []

    # Callback para guardar checkpoints
    if checkpoint_dir:
        checkpoint_filepath = os.path.join(checkpoint_dir, 'ckpt_{epoch}')
        checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
            filepath=checkpoint_filepath,
            save_weights_only=True,
            save_best_only=True,  # Guarda solo el mejor modelo
            monitor="loss",  # Basado en la pérdida
            mode="min",
            verbose=1
        )
        callbacks.append(checkpoint_callback)

        # Restaurar último checkpoint si existe
        checkpoint_latest = tf.train.latest_checkpoint(checkpoint_dir)
        if checkpoint_latest:
            model.load_weights(checkpoint_latest)
            initial_epoch = int(os.path.basename(checkpoint_latest).split('_')[-1])
            print(f'✅ Checkpoint encontrado: {checkpoint_latest}. Continuando desde la época {initial_epoch}.')
        else:
            initial_epoch = 0
    else:
        initial_epoch = 0


    # Entrenar el modelo
    history = model.fit(
        dataset,
        epochs=train_epochs,
        validation_data=validation_dataset,
        initial_epoch=initial_epoch,
        steps_per_epoch=train_steps,
         callbacks=[History()],
        verbose=2
    )

    return history, callbacks

In [84]:
from transformers import TFAutoModelForCausalLM, AutoModelForCausalLM
import tensorflow as tf
import torch

def load_gpt2_model(model_checkpoint, checkpoint_dir, dp, pytorch=False):
    """Carga un modelo GPT-2 con pesos desde el último checkpoint."""

    if pytorch:
        # Cargar modelo en PyTorch
        prediction_model = AutoModelForCausalLM.from_pretrained(model_checkpoint)
        print("✅ Modelo PyTorch base cargado correctamente.")
        return prediction_model  # No compila ni carga checkpoints en PyTorch

    # Cargar modelo en TensorFlow
    prediction_model = TFAutoModelForCausalLM.from_pretrained(model_checkpoint)
    
    if dp:
        noise_multiplier = 0.4272711315437153
        l2_norm_clip = 1e-6
        num_microbatches = BATCH_SIZE
        optimizer = DPKerasSGDOptimizer(
            learning_rate=0.03,
            noise_multiplier=noise_multiplier,
            l2_norm_clip=l2_norm_clip,
            num_microbatches=1
        )
        print("🔒 Usando DP-SGD con noise_multiplier =", noise_multiplier)
    else:
        optimizer = tf.keras.optimizers.SGD(learning_rate=0.03)
    
    # Cargar los pesos del último checkpoint si existe
    latest_checkpoint = tf.train.latest_checkpoint(checkpoint_dir)
    if latest_checkpoint:
        prediction_model.load_weights(latest_checkpoint)
        print(f'✅ Pesos cargados desde {latest_checkpoint}')
    else:
        print('⚠️ No se encontró ningún checkpoint, cargando pesos preentrenados.')

    # Compilar el modelo con la pérdida adecuada para modelos de lenguaje
    prediction_model.compile(optimizer=optimizer, loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))

    return prediction_model


In [85]:
from tensorflow.nn import softmax
import numpy as np
import tensorflow as tf

def compute_perplexity_for_secret(secrets, checkpoint_dir, vocab_size, dp=False):
    """Get perplexity for given examples.

    Args:
        secrets: a list, where each element is a secret dictionary
        checkpoint_dir: the directory from which to load checkpoints
        vocab_size: vocabulary size

    Returns:
        Perplexity for the input examples.
    """
    prediction_model = load_gpt2_model("gpt2", checkpoint_dir, dp)

    cce = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE)
    ppls = [{} for _ in range(len(secrets))]
    ppls_reference = [None] * len(secrets)
    
    for i, secret in enumerate(secrets):
        print("i:", i)
        for r in secret.datasets.keys():
            d = secret.datasets[r]
            indexed_seqs = d['data']  # This is the _BatchDataset
            all_losses = []
            # Iterate over the dataset (which is a batch of sequences)
            for seq_batch in indexed_seqs:
                # seq_batch is a batch of sequences, iterate over each sequence in the batch
                for seq in seq_batch:
                    tokens = tf.convert_to_tensor([seq])  # Convert the sequence to a tensor and add batch dimension
                    
                    # Get model logits for the sequence
                    logits = prediction_model.predict(tokens).logits

                    # Apply softmax to get probabilities
                    predictions_prob = softmax(logits, axis=-1)
                
                    # Calculate the loss (log-probabilities) for each token in the sequence
                    loss = 0
                    for k in range(1, len(seq)):  # Skip the first token, as it's the start token
                        token = seq[k]  # Current token
                        prob = predictions_prob[0, k-1, token].numpy()  # Probability for token i given previous tokens
                        loss -= np.log2(prob)  # Sum of log2 probabilities

                    # Store the loss for the current sequence
                    all_losses.append(loss / len(seq))  # Average loss over the sequence length

            # Store the perplexity for this dataset
            print("LOSS:", all_losses)
            if r == 0:
                ppls_reference[i] = all_losses
            else:
                ppls[i][r] = all_losses

    return ppls, ppls_reference


# Run training and secret sharer on MIMIC-III dataset.

In [86]:
#@title Generate secrets
# Here we present just one example and you can try other combinations of patterns,
# vocabs and number of repetitions. For example, you may use other patterns and
# different vocabularies for them.
patterns = ['{}'*5,
            'Your doctor is ' + '{}',
            '{}{}-{}{}-{}',
            '{}' * 10]
vocabs = [
    [
  "Chai", "Chan", "Chen", "Chey", "Chia", "Chiu", "Choe", 
        "Choi", "Chou", "Chow", "Cino", "Clay", "Cobb", "Cock",
        "Cody", "Coke", "Cola", "Cole", "Colt", "Como", "Cook",
        "Cory", "Côté", "Cruz", "Cseh", "Dahl", "Dale", "Dali", 
        "Daly", "Dane", "Dana", "Deák", "Deen", "Dean", 
        "Deng", "Devi", "Dias", "Díaz", "Dick", "Díez", "Ding", 
        "Dinu", "Dior", "Dong", "Duda", "Duff", "Duke", "Dunn", 
        "Dürr", "Dyer", "Eads", "Eady", "Earl", "Eide", "Enns", 
        "Erős", "Ezra", "Falk", "Fava", "Fear", "Feld", "Fend",
        "Feng", "Fenn", "Fini", "Finn", "Ford", "Foss", "Foth", 
        "Frei", "Frye", "Gaál", "Gage", "Gale", "Gama", "Gang", 
        "Garb", "Gary", "Gass", "Gibb", "Gill", "Glas", "Glen", 
        "Gold", "Good", "Gore", "Gori", "Gott", "Graf", "Gray",
        "Grec", "Grey", "Gros", "Guan", "Gump", "Gynt", "Haak", 
        "Haan", "Haas", "Hahn", "Haig", "Hail", "Hale", "Hall",
        "Hamm", "Hart", "Hase", "Hass", "Hawk", "Head", "Hext", 
        "Hier", "Hill", "Hino", "Hoch", "Hoek", "Holm", "Holt",
        "Hood", "Hope", "Horn","Lång", "Lang", "Lara", "Lehr", 
        "Lenz", "Lean", "Leon", "Mayr", "Leor", "Levi", "Liao",
        "Lien", "Lind", "Lobo", "Long", "Losa", "Love", "Löwe", 
        "Lowe", "Lowk", "Luna", "Lund", "Lupo", "Lupu", "Lusk",
        "Lyle", "Lyne", "Lynn", "Lyon", "Lyan", "Lyun", "Maas",
        "Mach", "Mack", "Macc", "Macy", "Maes", "Mägi",
        "Maki", "Maka", "Malá", "Mały", "Malo", "Mann", 
        "Manz", "Mari", "Mark", "Masi", "Mata", "Maus", "Mein",
        "Meir", "Melo", "Merx", "Mets", "Metz", "Moen", "Monk",
        "Moon", "Muun", "Moun", "Mora", "Mori", "Moss", 
        "Mosy", "Moto", "Moya", "Muir", "Mutō", "Myer", "Nagi",
        "Nagy", "Najm", "Nana", "Nani", "Nash", "Nass", "Nave",
        "Neal", "Neil", "Nepi", "Neri", "Ness", "Nisi",
        "Noel", "Noel", "Noel", "Ohme", "Oláh", "Olmo", 
        "Öman", "Omar", "Otis", "Otto", "Owen", "Pace", 
        "Page", "Pain", "Pape", "Papp", "Papp", "Park",
        "Park", "Park", "Pärn", "Paul", "Peak", "Peck",
        "Peck", "Peel", "Peña", "Peng", "Penn", "Penn", 
        "Pető", "Phạm", "Phan", "Pini", "Pitt", "Pohl", 
        "Pohl", "Pond", "Pons", "Popa", "Pope", "Post", 
        "Puga", "Pugh", "Puig", "Rácz", "Radu", "Rais",
        "Rake", "Rana", "Rapp", "Rapp", "Rask", "Ravn",
        "Read", "Read", "Reed", "Rees", "Řeha", "Reid", 
        "Reis", "Rémy", "Rhee", "Rice", "Rico", "Rigó", 
        "Ríos", "Rios", "Riva", "Roca", "Roig", "Rojo", 
        "Roma", "Romà", "Rome", "Rosa", "Rose", "Rose",
        "Ross", "Roth", "Roux", "Rowe", "Rowe", "Ruan",
        "Ruff", "Ruiz", "Rush", "Rusu", "Ruud", "Ryan",
        "Ryba", "Ryer", "Saab", "Saad", "Saar", "Sáez",
        "Sala", "Salo", "Sams","Sano", "Sanz", "Sass", 
        "Sato", "Satō", "Seki", "Sepp", "Sgro", "Shah",
        "Shaw", "Shaw", "Shea", "Shen", "Shin", "Sims", 
        "Sitz", "Sitz", "Skov", "Smit", "Snel", "Solo",
        "Song", "Sorg", "Sosa", "Soto","Soun"

]
]* 4
num_repetitions = [1000, 10000, 100000] 
num_secrets_for_repetitions = [10] * len(num_repetitions)
num_references = 300 #65536
secret_configs = [SecretConfig(num_repetitions, num_secrets_for_repetitions, num_references,
                               properties=TextSecretProperties(vocab, pattern))
                  for vocab, pattern in zip(vocabs, patterns)]
secrets = generate_text_secrets_and_references(secret_configs)

# Let's look at the variable "secrets"
print(f'"secrets" is a list and the length is {len(secrets)} because we have four sets of secrets.')
print(f'The type of the element in "secrets" is {type(secrets[0])}.')
print(f'The field "config" contains its configuration: {secrets[0].config}')
print(f'The filed "secrets" is a dictionary, with key being the number of repetitions and value being a list of secrets.',
      f'For example, the secrets that are going to be repeated for 1000 times are {secrets[0].secrets[1000]}.')
print(f'The field "references" is a list of reference sequences that are not going to be injected. The first 10 elements are {secrets[0].references[:10]}.')

"secrets" is a list and the length is 4 because we have four sets of secrets.
The type of the element in "secrets" is <class 'tensorflow_privacy.privacy.privacy_tests.secret_sharer.generate_secrets.SecretsSet'>.
The field "config" contains its configuration: SecretConfig(num_repetitions=[1000, 10000, 100000], num_secrets_for_repetitions=[10, 10, 10], num_references=300, name='', properties=TextSecretProperties(vocab=['Chai', 'Chan', 'Chen', 'Chey', 'Chia', 'Chiu', 'Choe', 'Choi', 'Chou', 'Chow', 'Cino', 'Clay', 'Cobb', 'Cock', 'Cody', 'Coke', 'Cola', 'Cole', 'Colt', 'Como', 'Cook', 'Cory', 'Côté', 'Cruz', 'Cseh', 'Dahl', 'Dale', 'Dali', 'Daly', 'Dane', 'Dana', 'Deák', 'Deen', 'Dean', 'Deng', 'Devi', 'Dias', 'Díaz', 'Dick', 'Díez', 'Ding', 'Dinu', 'Dior', 'Dong', 'Duda', 'Duff', 'Duke', 'Dunn', 'Dürr', 'Dyer', 'Eads', 'Eady', 'Earl', 'Eide', 'Enns', 'Erős', 'Ezra', 'Falk', 'Fava', 'Fear', 'Feld', 'Fend', 'Feng', 'Fenn', 'Fini', 'Finn', 'Ford', 'Foss', 'Foth', 'Frei', 'Frye', 'Gaál', 'Ga

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
for i in range(len(secrets)):
  train_dataset, validation_dataset = get_dataset(
      [secrets[i]],
      batch_size=BATCH_SIZE)
    
  checkpoint_dir = 'DP-checkpoints'+str(i)
  print("dataset", i, "loaded")
  history, callbacks = train_model(train_dataset, validation_dataset, batch_size=32, vocab_size=tokenizer.vocab_size, train_epochs=TRAIN_EPOCHS, checkpoint_dir=checkpoint_dir, dp=True)

In [ ]:
#@title Secret sharer evaluation
ppls_all, ppls_reference_all = [], []
for i in range(len(secrets)):
  checkpoint_dir =  'checkpoints'+str(i)
  ppls, ppls_reference = compute_perplexity_for_secret(
      [secrets[1]], checkpoint_dir, tokenizer.vocab_size)
  ppls_all += ppls
  ppls_reference_all += ppls_reference


for i, (vocab, pattern, p, p_r) in enumerate(zip(vocabs, patterns, ppls_all, ppls_reference_all)):
  print("p", p, "p_r",p_r)
  exposure_interpolation = compute_exposure_interpolation(p, p_r)

  exposure_extrapolation = compute_exposure_extrapolation(p, p_r)

  # exposure_interpolation (or exposure_extrapolation) is a dictionary, where the key is the number of repetitions and
  # the value is a list of exposures (not necessarily in the same order as the secrets). We will now take a look at
  # the average exposure (with standard deviation).
  print(f'Secrets #{i+1}, vocab={vocab}, pattern={pattern}')
  print('  Interpolation:', '; '.join([f'repetition={r}, avg_exposure={np.mean(exposure_interpolation[r]):.2f}±{np.std(exposure_interpolation[r]):.2f}' for r in num_repetitions]))
  print('  Extrapolation:', '; '.join([f'repetition={r}, avg_exposure={np.mean(exposure_extrapolation[r]):.2f}±{np.std(exposure_extrapolation[r]):.2f}' for r in num_repetitions]))

# Comparing models

In [89]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

train_dataset, validation_dataset = get_dataset(
  [secrets[1]],
  batch_size=BATCH_SIZE)

#checkpoint_dir = 'DP-checkpoints'
#history, callbacks = train_model(dataset, batch_size=32, vocab_size=tokenizer.vocab_size, train_epochs=TRAIN_EPOCHS, checkpoint_dir=checkpoint_dir, dp=True)

checkpoint_dir = 'checkpoints'
history, callbacks = train_model(train_dataset, validation_dataset, batch_size=32, vocab_size=tokenizer.vocab_size, train_epochs=TRAIN_EPOCHS, checkpoint_dir=checkpoint_dir, dp=False)

Token indices sequence length is longer than the specified maximum sequence length for this model (5808277 > 1024). Running this sequence through the model will result in indexing errors
All PyTorch model weights were used when initializing TFGPT2LMHeadModel.

All the weights of TFGPT2LMHeadModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFGPT2LMHeadModel for predictions without further training.


Epoch 1/2
6/6 - 2728s - loss: 81.5183 - val_loss: 54.3375 - 2728s/epoch - 455s/step
Epoch 2/2
6/6 - 2701s - loss: 25.9208 - val_loss: 12.2667 - 2701s/epoch - 450s/step


In [ ]:
#@title Comparing exposure between base model, DP-SGD and SGD

#SGD model:
ppls_all, ppls_reference_all = [], []
checkpoint_dir =  'checkpoints'
ppls, ppls_reference = compute_perplexity_for_secret(
  [secrets[1]], checkpoint_dir, tokenizer.vocab_size)
ppls_all += ppls
ppls_reference_all += ppls_reference


for i, (vocab, pattern, p, p_r) in enumerate(zip(vocabs, patterns, ppls_all, ppls_reference_all)):
  if (i != 0):
      break
  else:
      exposure_interpolation = compute_exposure_interpolation(p, p_r)
    
      exposure_extrapolation = compute_exposure_extrapolation(p, p_r)
    
      # exposure_interpolation (or exposure_extrapolation) is a dictionary, where the key is the number of repetitions and
      # the value is a list of exposures (not necessarily in the same order as the secrets). We will now take a look at
      # the average exposure (with standard deviation).
      print(f'Secrets #{i+1}, vocab={vocab}, pattern={pattern}')
      print('  Interpolation:', '; '.join([f'repetition={r}, avg_exposure={np.mean(exposure_interpolation[r]):.2f}±{np.std(exposure_interpolation[r]):.2f}' for r in num_repetitions]))
      print('  Extrapolation:', '; '.join([f'repetition={r}, avg_exposure={np.mean(exposure_extrapolation[r]):.2f}±{np.std(exposure_extrapolation[r]):.2f}' for r in num_repetitions]))

#DP-SGD model:
ppls_all, ppls_reference_all = [], []
checkpoint_dir =  'DP-checkpoints'
ppls, ppls_reference = compute_perplexity_for_secret(
  [secrets[1]], checkpoint_dir, tokenizer.vocab_size, dp=True)
ppls_all += ppls
ppls_reference_all += ppls_reference


for i, (vocab, pattern, p, p_r) in enumerate(zip(vocabs, patterns, ppls_all, ppls_reference_all)):
  if (i != 0):
      break
  else:
      exposure_interpolation = compute_exposure_interpolation(p, p_r)
    
      exposure_extrapolation = compute_exposure_extrapolation(p, p_r)
    
      # exposure_interpolation (or exposure_extrapolation) is a dictionary, where the key is the number of repetitions and
      # the value is a list of exposures (not necessarily in the same order as the secrets). We will now take a look at
      # the average exposure (with standard deviation).
      print(f'Secrets #{i+1}, vocab={vocab}, pattern={pattern}')
      print('  Interpolation:', '; '.join([f'repetition={r}, avg_exposure={np.mean(exposure_interpolation[r]):.2f}±{np.std(exposure_interpolation[r]):.2f}' for r in num_repetitions]))
      print('  Extrapolation:', '; '.join([f'repetition={r}, avg_exposure={np.mean(exposure_extrapolation[r]):.2f}±{np.std(exposure_extrapolation[r]):.2f}' for r in num_repetitions]))
#base model (GPT2 without finetuning):
ppls_all, ppls_reference_all = [], []
checkpoint_dir =  'base'
ppls, ppls_reference = compute_perplexity_for_secret(
  [secrets[1]], checkpoint_dir, tokenizer.vocab_size)
ppls_all += ppls
ppls_reference_all += ppls_reference


for i, (vocab, pattern, p, p_r) in enumerate(zip(vocabs, patterns, ppls_all, ppls_reference_all)):
  if (i != 0):
      break
  else:
      exposure_interpolation = compute_exposure_interpolation(p, p_r)
    
      exposure_extrapolation = compute_exposure_extrapolation(p, p_r)
    
      # exposure_interpolation (or exposure_extrapolation) is a dictionary, where the key is the number of repetitions and
      # the value is a list of exposures (not necessarily in the same order as the secrets). We will now take a look at
      # the average exposure (with standard deviation).
      print(f'Secrets #{i+1}, vocab={vocab}, pattern={pattern}')
      print('  Interpolation:', '; '.join([f'repetition={r}, avg_exposure={np.mean(exposure_interpolation[r]):.2f}±{np.std(exposure_interpolation[r]):.2f}' for r in num_repetitions]))
      print('  Extrapolation:', '; '.join([f'repetition={r}, avg_exposure={np.mean(exposure_extrapolation[r]):.2f}±{np.std(exposure_extrapolation[r]):.2f}' for r in num_repetitions]))

In [ ]:
%pip install datasets

In [ ]:
%pip install torch

In [92]:
%pip install immutabledict

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [94]:
%pip install statsmodels

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 31.3 MB/s eta 0:00:00 0:00:01
Note: you may need to restart the kernel to use updated packages.


## Membership attacks

In [96]:
import tensorflow_privacy.privacy.privacy_tests.membership_inference_attack.membership_inference_attack as mia
from tensorflow_privacy.privacy.privacy_tests.membership_inference_attack.data_structures import AttackInputData
from tensorflow_privacy.privacy.privacy_tests.membership_inference_attack.data_structures import SlicingSpec
from tensorflow_privacy.privacy.privacy_tests.membership_inference_attack.data_structures import AttackType

train_loss = np.array(history.history["loss"])  
val_loss = np.array(history.history["val_loss"])

attacks_result = mia.run_attacks(
    AttackInputData(loss_train=train_loss, loss_test=val_loss)
)
print(attacks_result.summary())

ValueError: loss_train should be a numpy array.

In [ ]:
import tensorflow_privacy.privacy.membership_inference_attack.plotting as plotting
plotting.plot_roc_curve(attacks_result.get_result_with_max_auc().roc_curve)